In [6]:
import json
import os

import agama
import matplotlib
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

agama.setUnits(mass=1, length=1, velocity=1)
units = agama.getUnits()

In [2]:
dat_dir = "../data/"
mw_gcs_fil = "mw_gcs/mw_gcs.csv"
mw_gcs = pd.read_csv(dat_dir + mw_gcs_fil)

In [3]:
pot_dir = "../.venv312/lib/python3.12/site-packages/agama/data/"
pot_type = "PriceWhelan22.ini"

pot_file = pot_dir + pot_type
pot = agama.Potential(pot_file)
af = agama.ActionFinder(pot, interp=False)

In [9]:
pxyz = np.column_stack((mw_gcs["x_gc"], mw_gcs["y_gc"], mw_gcs["z_gc"]))
vxyz = np.column_stack((mw_gcs["u"], mw_gcs["v"], mw_gcs["w"]))

posvel = np.column_stack((pxyz, vxyz))
result = agama.orbit(potential=pot, ic=posvel, time=10 * pot.Tcirc(posvel), trajsize=1000)

t_orb = []
for result_i in result:
    t_i_agama = result_i[0]
    t_i_myr = t_i_agama * units["time"]  # Myr
    orb_i = result_i[1]

    r_i = np.linalg.norm(orb_i[:, 0:3], axis=1)
    peaks, _ = find_peaks(r_i)

    if len(peaks) < 2:
        t_orb_i = np.nan
    else:
        t_orb_i = np.mean(np.diff(t_i_myr[peaks]))

    t_orb.append(t_orb_i)
t_orb = np.array(t_orb)


165 orbits complete (7500 orbits/s)


In [13]:
mw_gcs["t_orb"] = t_orb

mw_gcs.to_csv(dat_dir + mw_gcs_fil, index=False)